In [50]:
import torch
import math

device = 'cuda' if torch.cuda.is_available() else 'cpu'

batch_size, seq_len, emb_dim = 2, 5, 32

In [87]:
class SelfAttn(torch.nn.Module):
  def __init__(self, emb_dim=32, num_heads=4):
    super().__init__()
    self.head_dim = emb_dim // num_heads
    self.emb_dim = emb_dim
    self.num_heads = num_heads

    self.attn = torch.nn.Linear(emb_dim, emb_dim * 3)
    self.output_proj = torch.nn.Linear(emb_dim, emb_dim)

  def forward(self, x):
    x = self.attn(x)
    q, k, v = torch.split(x, self.emb_dim, dim=-1)
    print(q.size())
    
    q = q.reshape(batch_size, seq_len, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
    k = k.reshape(batch_size, seq_len, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
    v = v.reshape(batch_size, seq_len, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
    
    attn_weights = (q @ k.transpose(-1, -2)) / math.sqrt(self.emb_dim)
    
    mask = torch.ones(seq_len, seq_len, dtype=torch.bool).tril(diagonal=0)
    attn_weights = torch.where(mask == True, attn_weights, float('-inf'))
    
    attn_weights = torch.nn.functional.softmax(attn_weights)
    
    attn_result = attn_weights @ v
    
    attn_result = attn_result.permute(0, 2, 1, 3).reshape(batch_size, seq_len, self.emb_dim)
    
    return self.output_proj(attn_result)
  

x = torch.randn(batch_size, seq_len, emb_dim).to(device)
SelfAttn(emb_dim=emb_dim).to(device)(x).size()

torch.Size([2, 5, 32])


/var/folders/7l/3qdkg7n17mj7g3w_q8yf5xzc0000gq/T/ipykernel_17053/4125806331.py:25: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  attn_weights = torch.nn.functional.softmax(attn_weights)


torch.Size([2, 5, 32])

torch.Size([2, 5])